# Level 5A — đánh giá lại 3 seed 42, 44, 46 bằng cùng Beam Search với Level 1

Notebook này **chỉ đọc ba model L5A đã train đủ 15 epoch** và đánh giá lại trên 5.000 ảnh COCO Karpathy Test. Không train lại và không ghi lên checkpoint, cache hay kết quả cũ. Model gốc là kiến trúc **5 knowledge tokens**. Cả ba seed đã fine-tune từ cùng L4 `full/Lan01_seed45`; kết quả không tách riêng tác dụng Semantic/MMR khỏi phần fine-tuning.

Lý do đánh giá lại: decoder cũ và decoder Lv1 mới cùng ghi beam=3/alpha=0.6 nhưng xử lý EOS khác nhau. Bản này dùng **normalized_candidates_v1** giống Lv1: xếp hạng cả beam đã có EOS và beam đang mở rộng ở mỗi bước; `max_len=30` gồm BOS. Điểm mới có thể tăng hoặc giảm. Giữ nguyên `w=0.6, lambda=0.7` đã chọn trên Val của mỗi model, không dùng Test để chọn tham số hoặc chọn seed.

**Chạy:** bật GPU Colab → chạy Cell 1 → ở Cell 2 chọn 42 → chạy lần lượt Cell 2–6. Xong đổi sang 44 rồi 46 và chạy lại Cell 2–6. Với runtime mới chạy lại từ Cell 1. Giữ cùng `EXPERIMENT_TAG` cho ba seed. Cell 6 chỉ tổng hợp khi đủ ba kết quả mới.

Kết quả mới nằm ở `baseline_level5/lv5_A/multiseed_beam3_v1/run_<hash>/seed_42`, `seed_44`, `seed_46`. Mỗi seed chỉ mở **một phiên Colab**. Khi bị ngắt, chạy lại cùng tag/seed để tiếp tục. Chỉ bật `OLD_SESSION_STOPPED` khi đã xác nhận phiên cũ dừng hẳn. File hỏng/khác cấu hình được giữ nguyên và notebook dừng để kiểm tra.

**Cache cũ:** đã xác minh byte SHA của cache Test từng seed trên Drive. Notebook ưu tiên đường dẫn gốc; nếu Drive hiện thư mục trùng tên có hậu tố thì chỉ nhận bản khớp SHA, contract và seed. Không yêu cầu hoặc tạo file `.sha256`; nếu sidecar có sẵn thì kiểm tra. Không nhìn thấy bản cache đúng sẽ dừng trước khi tạo output.

Metric dùng pycocoevalcap 1.2/PTB/Java 11: BLEU-1..4, METEOR, ROUGE-L, CIDEr, SPICE. JSON lưu thang gốc; bảng hiển thị ×100. Std mẫu (`ddof=1`, n=3) mô tả biến thiên giữa seed, không tự chứng minh ý nghĩa thống kê. Notebook đã kiểm tra tĩnh và kiểm thử nhỏ; chưa chạy toàn bộ ba seed trên GPU Colab. Công cụ cài đặt/cache chấm điểm chỉ nằm trong `/content` của runtime.


In [1]:
#@title CELL 1 — Môi trường chấm điểm trên Colab GPU
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, subprocess, shutil, tempfile, tarfile, urllib.request, zipfile
from pathlib import Path
from importlib import metadata

RUNTIME_DIR = Path("/content/level5a_beam3_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pycocoevalcap==1.2", "nltk>=3.9,<4", "tqdm>=4.66,<5"],
    check=True,
)

def java11_version(java):
    result = subprocess.run([str(java), "-version"], capture_output=True, text=True)
    return result.returncode == 0 and 'version "11.' in result.stderr

java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
if not any(java11_version(p) for p in java_candidates):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq",
                        "openjdk-11-jre-headless"], check=True)
    except subprocess.CalledProcessError:
        # Colab Ubuntu mới có thể không cung cấp package OpenJDK 11.
        java_root = RUNTIME_DIR / "java11"
        java_root.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            archive = Path(tmp) / "jdk11.tar.gz"
            urllib.request.urlretrieve(
                "https://api.adoptium.net/v3/binary/latest/11/ga/linux/x64/jre/hotspot/normal/eclipse",
                archive,
            )
            with tarfile.open(archive, "r:gz") as tar:
                tar.extractall(java_root, filter="data")
    java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
    java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
JAVA_BIN = next((p for p in java_candidates if java11_version(p)), None)
if JAVA_BIN is None:
    raise RuntimeError("Không cài được Java 11; xem log cài đặt ở trên.")
os.environ["JAVA_HOME"] = str(JAVA_BIN.parent.parent)
os.environ["PATH"] = str(JAVA_BIN.parent) + os.pathsep + os.environ["PATH"]
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

import nltk
for resource in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f"Không tải được NLTK resource: {resource}")

import pycocoevalcap.spice.spice as spice_module
SPICE_DIR = Path(spice_module.__file__).parent
SPICE_LIB = SPICE_DIR / "lib"
SPICE_LIB.mkdir(exist_ok=True)
# Tải cả code jar và models jar qua HTTPS, tránh downloader HTTP cũ.
for suffix in ("", "-models"):
    jar_name = f"stanford-corenlp-3.6.0{suffix}.jar"
    jar_path = SPICE_LIB / jar_name
    if not jar_path.exists() or not zipfile.is_zipfile(jar_path):
        url = ("https://repo.maven.apache.org/maven2/edu/stanford/nlp/"
               f"stanford-corenlp/3.6.0/{jar_name}")
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            downloaded = Path(tmp) / jar_name
            urllib.request.urlretrieve(url, downloaded)
            if not zipfile.is_zipfile(downloaded):
                raise RuntimeError(f"Jar tải về không hợp lệ: {jar_name}")
            shutil.copyfile(downloaded, jar_path)

# Package và cache SPICE đều nằm trong runtime; không ghi công cụ lên Drive.
spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

class ManagedMeteor(Meteor):
    def close(self):
        process = getattr(self, "meteor_p", None)
        if process is not None and process.poll() is None:
            try:
                if process.stdin is not None:
                    process.stdin.close()
            except (OSError, ValueError):
                pass
            finally:
                process.kill()
                process.wait(timeout=30)
    def __del__(self):
        self.close()

def close_meteor(scorer):
    scorer.close()

_smoke_gts = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_smoke_res = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_meteor = ManagedMeteor()
try:
    _meteor_score, _ = _meteor.compute_score(_smoke_gts, _smoke_res)
finally:
    close_meteor(_meteor)
del _meteor
_spice_score, _ = Spice().compute_score(_smoke_gts, _smoke_res)
import math
if not all(math.isfinite(float(x)) for x in (_meteor_score, _spice_score)):
    raise RuntimeError("Smoke test METEOR/SPICE thất bại.")
print(f"Bộ chấm điểm OK • METEOR={_meteor_score:.4f} • SPICE={_spice_score:.4f}")
print("pycocoevalcap", metadata.version("pycocoevalcap"), "| NLTK", nltk.__version__)
# Nguồn triển khai: https://github.com/salaniz/pycocoevalcap


openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)

Bộ chấm điểm OK • METEOR=1.0000 • SPICE=1.0000
pycocoevalcap 1.2 | NLTK 3.9.1


In [7]:
#@title CELL 2 — Chọn seed và kiểm tra toàn bộ nguồn cũ (chỉ đọc)
SEED_TO_RUN = "44" #@param ["42", "44", "46"]
OLD_SESSION_STOPPED = False #@param {type:"boolean"}
EXPERIMENT_TAG = "main" #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import json, re, hashlib, platform, time, gc, math
from collections import defaultdict
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import display, Markdown
sys.dont_write_bytecode = True
SELECTED_SEED = int(SEED_TO_RUN)
ALL_SEEDS = [42, 44, 46]
if SELECTED_SEED not in ALL_SEEDS:
    raise ValueError("Chỉ chọn seed 42, 44 hoặc 46.")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}", EXPERIMENT_TAG):
    raise ValueError("Tag chỉ gồm chữ không dấu, số, gạch dưới/gạch ngang, tối đa 40 ký tự.")
DATA_ROOT = Path("/content/drive/MyDrive/DataKhoaLuan/Data")
SOURCE_ROOT = DATA_ROOT / "baseline_level5/lv5_A/trained_5tokens_v1"
OUTPUT_ROOT = DATA_ROOT / "baseline_level5/lv5_A/multiseed_beam3_v1"
FEATURE_DIR = DATA_ROOT / "baseline_level1/clip_image_features"
KB_DIR = DATA_ROOT / "knowledge_base_level1_karpathy"
MAX_LEN, PAD_ID, BOS_ID, EOS_ID, UNK_ID = 30, 0, 1, 2, 3
METRIC_NAMES = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr", "SPICE"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Bật GPU trong Runtime → Change runtime type trước khi đánh giá 5.000 ảnh.")

def read_json(path):
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def canonical_hash(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""):
            h.update(block)
    return h.hexdigest()

SOURCE_FILE_HASHES, SOURCE_FILE_PATHS = {}, {}
def register_source(path, logical_key=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu nguồn cũ; không tự tạo hoặc thay thế: {path}")
    # Folder trùng tên trên Drive có thể mang hậu tố khác nhau giữa các VM.
    # Protocol dùng tên logic ổn định; kiểm tra nguồn vẫn đọc đúng file thực tế.
    key = path.relative_to(DATA_ROOT).as_posix() if logical_key is None else logical_key
    print("Kiểm tra SHA-256:", key, flush=True)
    SOURCE_FILE_HASHES[key] = sha256_file(path)
    SOURCE_FILE_PATHS[key] = path
    return path

vocab_path = register_source(DATA_ROOT / "baseline_level1/vocab.json")
vocab = read_json(vocab_path)
stoi = {str(k): int(v) for k,v in vocab["stoi"].items()}
raw_itos = vocab["itos"]
itos = [str(x) for x in raw_itos] if isinstance(raw_itos,list) else [str(raw_itos[str(i)]) for i in range(len(stoi))]
VOCAB_SIZE = len(stoi)
assert VOCAB_SIZE == 16214 and sorted(stoi.values()) == list(range(VOCAB_SIZE))
assert all(stoi.get(t)==i for i,t in enumerate(("<PAD>","<BOS>","<EOS>","<UNK>")))
assert all(itos[i]==t for t,i in stoi.items()) and int(vocab.get("max_length",MAX_LEN))==MAX_LEN
ann_candidates = [DATA_ROOT/"COCO_2014/annotations_trainval2014/annotations", DATA_ROOT/"COCO_2014/annotations"]
valid_annotations = [p for p in ann_candidates if (p/"captions_karpathy_test.json").is_file()]
if len(valid_annotations) != 1:
    raise RuntimeError(f"Nguồn annotations Test phải duy nhất: {valid_annotations}")
annotation_path = register_source(valid_annotations[0]/"captions_karpathy_test.json")
annotations = read_json(annotation_path)
references = defaultdict(list)
for row in annotations["annotations"]:
    if not isinstance(row["caption"], str) or not row["caption"].strip():
        raise ValueError("Reference rỗng/sai kiểu; giữ nguồn và dừng.")
    references[int(row["image_id"])].append(row["caption"])
references = dict(references)
TEST_IDS = sorted(references)
assert len(TEST_IDS)==5000 and sum(map(len,references.values()))==25010
assert {int(row["id"]) for row in annotations["images"]}==set(TEST_IDS)
image_ids = {"test": TEST_IDS}
del annotations
feature_path = register_source(FEATURE_DIR/"test_features.npy")
feature_ids_path = register_source(FEATURE_DIR/"test_image_ids.json")
feature_ids = [int(x) for x in read_json(feature_ids_path)]
assert len(feature_ids)==len(set(feature_ids))==5000 and set(feature_ids)==set(TEST_IDS)
raw_features = np.load(feature_path, allow_pickle=False)
assert raw_features.shape==(5000,512) and raw_features.dtype==np.float32 and np.isfinite(raw_features).all()
assert np.allclose(np.linalg.norm(raw_features,axis=1),1.,atol=1e-3)
feature_positions = {iid:i for i,iid in enumerate(feature_ids)}
test_features = np.array(raw_features[[feature_positions[i] for i in TEST_IDS]],dtype=np.float32,copy=True)
del raw_features
train_ids_path = register_source(FEATURE_DIR/"train_image_ids.json")
train_ids = set(map(int, read_json(train_ids_path)))
assert len(train_ids)==113287 and not (train_ids & set(TEST_IDS))
kb_path = register_source(KB_DIR/"text_embeddings_train_restval.npy")
kb_mapping_path = register_source(KB_DIR/"text_embedding_mapping.jsonl")
kb = np.load(kb_path, mmap_mode="r", allow_pickle=False)
assert kb.shape==(566747,512) and kb.dtype==np.float32
kb_source_ids = []
with kb_mapping_path.open(encoding="utf-8") as f:
    for position,line in enumerate(f):
        row=json.loads(line)
        assert int(row.get("row_id",position))==position and row["split"] in {"train","restval"}
        iid=int(row["image_id"])
        assert iid in train_ids and str(row["caption"]).strip()
        kb_source_ids.append(iid)
assert len(kb_source_ids)==len(kb) and set(kb_source_ids)==train_ids
kb_source_ids=np.asarray(kb_source_ids,dtype=np.int64)

EXPECTED_STATES = {
    42:"b5c68614e3ef5de6c520a8d6ef020db44266deb1f3cd24f7d3640f38a1cb7e19",
    44:"7b6d54d7251f872306e18333db4ad8cad0bcdae9d50b0a37e31afaff8d3ed2f1",
    46:"18b372e6fa6ad2d4f7adabdf20e09de90c405bd078af996121980929f4b1b8c3",
}
EXPECTED_INIT_FILE="c9341fdf1b5f67559660186bfe179e75ae9691de49aea4fe9a6f67cfa19a49e6"
EXPECTED_INIT_STATE="703eb5e24b231e761f8362773f3c9ebb71183a1fc0e80b01e8cd801dc19c0730"
MEMORY_CONTRACT = {
    "retrieved_captions":5,"tokens_per_caption":1,"knowledge_memory_tokens":5,
    "caption_embedding_dim":512,"d_model":256,"projection":"Linear(512,256)",
    "gate_input":"5 caption embeddings and 5 normalized relevance scores",
    "warmstart_rule":"strict_native5_no_transformation",
}

def original_input_hash(manifest, relative):
    values=[v for k,v in manifest["input_sha256"].items() if str(k).replace("\\","/").endswith("/"+relative)]
    if len(values)!=1:
        raise ValueError(f"Manifest nguồn thiếu/không duy nhất fingerprint: {relative}")
    return values[0]

def cache_payload_hash(arrays, metadata):
    h=hashlib.sha256(canonical({k:v for k,v in metadata.items() if k!="cache_sha256"}).encode())
    for name in sorted(arrays):
        value=np.ascontiguousarray(arrays[name])
        h.update(canonical([name,value.dtype.str,list(value.shape)]).encode())
        h.update(value.tobytes())
    return h.hexdigest()

# Byte SHA đã đối chiếu với đúng cache Test của từng seed trên Drive.
PINNED_TEST_CACHE_SHA256 = {
    42: "81346729a4097443d712e88a9e27b23f1258b905fc3286a47b2f2473d66a1184",
    44: "26592101914f01cb4cf694083a035848eaab4eba2b2889e18da54ed998dc6e2e",
    46: "2dfad1402d44df501bca86ebb45fd472d5d31d3c4c24ec620a35238be6cb91a7",
}

def resolve_original_test_cache(seed, contract):
    """Chỉ đọc: ưu tiên đường dẫn cũ; fallback cùng prefix khi byte SHA khớp."""
    if seed not in PINNED_TEST_CACHE_SHA256 or not isinstance(contract, str) or not re.fullmatch(r"[0-9a-f]{64}", contract):
        raise ValueError("Seed/contract cache nguồn không hợp lệ.")
    expected = PINNED_TEST_CACHE_SHA256[seed]
    cache_root = SOURCE_ROOT / "cache"
    prefix = contract[:20]
    direct = cache_root / prefix / "w0.6_lambda0.7/test.npz"
    checked, visible_files = set(), 0
    def exact_match(candidate):
        nonlocal visible_files
        key = str(candidate)
        if key in checked:
            return False
        checked.add(key)
        if not candidate.is_file():
            return False
        visible_files += 1
        return sha256_file(candidate) == expected
    if exact_match(direct):
        return direct
    if cache_root.is_dir():
        # Chỉ duyệt thư mục anh em cùng prefix20; có thể có hậu tố do Drive FUSE.
        # Không nhận cache của seed/contract khác và không chọn file đầu tiên tùy ý.
        for folder in sorted(cache_root.iterdir(), key=lambda path: path.name):
            if folder.name.startswith(prefix) and folder.is_dir():
                candidate = folder / "w0.6_lambda0.7/test.npz"
                if exact_match(candidate):
                    print(f"Seed {seed}: dùng bản cache trùng tên đã khớp SHA:", candidate)
                    return candidate
    raise FileNotFoundError(
        f"Không thấy cache Test seed {seed} đúng SHA đã xác minh (đã kiểm tra {visible_files} file). "
        "Drive có thể đang ẩn/chưa đồng bộ thư mục trùng tên. Chưa tạo output mới; "
        "không xóa/đổi tên/sửa cache cũ. Kiểm tra thư mục cache rồi chạy lại Cell 2.")

SOURCE_RECORDS, SOURCE_CACHES = {}, {}
common_inputs=[vocab_path,annotation_path,feature_path,feature_ids_path,train_ids_path,kb_path,kb_mapping_path]
for seed in ALL_SEEDS:
    work=SOURCE_ROOT/f"seed{seed}"
    paths={name:register_source(work/name) for name in ["run_manifest.json","training_log.json","final_config.json","best_model.pt"]}
    manifest,log,cfg=[read_json(paths[name]) for name in ["run_manifest.json","training_log.json","final_config.json"]]
    contract=manifest["contract_sha256"]
    assert manifest["seed"]==log["seed"]==cfg["seed"]==seed
    assert log["contract_sha256"]==cfg["contract_sha256"]==contract
    assert manifest["architecture"]==cfg["architecture"]=="level5_native5_trained_v1"
    assert manifest["memory_contract"]==cfg["memory_contract"]==MEMORY_CONTRACT
    assert manifest["init_checkpoint_sha256"]==cfg["init_checkpoint_sha256"]==EXPECTED_INIT_FILE
    assert log["complete"] is True and log["completed_epochs"]==15 and log["optimizer_steps"]==132840
    assert len(log["history"])==15 and cfg["selected_on"]=="validation_only"
    assert cfg["checkpoint_sha256"]==EXPECTED_STATES[seed]
    assert float(cfg["w"])==0.6 and float(cfg["lambda_param"])==0.7
    assert cfg["code_sha256"]==manifest["code_sha256"]==log["code_sha256"]
    for path in common_inputs:
        rel=path.relative_to(DATA_ROOT).as_posix()
        assert original_input_hash(manifest,rel)==SOURCE_FILE_HASHES[rel],f"Input đã đổi so với train seed {seed}: {rel}"
    logical_cache = SOURCE_ROOT/"cache"/contract[:20]/"w0.6_lambda0.7/test.npz"
    logical_cache_key = logical_cache.relative_to(DATA_ROOT).as_posix()
    cache_path = register_source(resolve_original_test_cache(seed, contract), logical_key=logical_cache_key)
    if SOURCE_FILE_HASHES[logical_cache_key] != PINNED_TEST_CACHE_SHA256[seed]:
        raise RuntimeError(f"Cache Test seed {seed} vừa đổi trong lúc đọc; chưa tạo output.")
    # Cache gốc của nhóm không có sidecar. Nếu đã có thì kiểm tra; không tạo thêm.
    sidecar = cache_path.with_suffix(".npz.sha256")
    if sidecar.exists():
        if not sidecar.is_file() or sidecar.read_text(encoding="ascii").strip() != PINNED_TEST_CACHE_SHA256[seed]:
            raise ValueError(f"Sidecar cache seed {seed} khác SHA đã xác minh; giữ nguyên và dừng.")
    with np.load(cache_path,allow_pickle=False) as z:
        cache_meta=json.loads(str(z["metadata_json"].item()))
        arrays={k:z[k] for k in z.files if k!="metadata_json"}
    assert cache_meta["cache_sha256"]==cache_payload_hash(arrays,cache_meta)
    assert cache_meta["seed"]==seed and cache_meta["contract_sha256"]==contract
    assert cache_meta["stage"]=="rerank_mmr" and cache_meta["split"]=="test"
    assert float(cache_meta["w"])==0.6 and float(cache_meta["lambda_param"])==0.7
    assert cache_meta["ids_sha256"]==canonical_hash(TEST_IDS)
    assert cache_meta["code_sha256"]==manifest["code_sha256"]
    ids,rows,scores=arrays["image_ids"],arrays["row_indices"],arrays["scores"]
    assert ids.tolist()==TEST_IDS and rows.shape==scores.shape==(5000,5)
    assert np.issubdtype(rows.dtype,np.integer) and np.all((rows>=0)&(rows<len(kb)))
    assert np.isfinite(scores).all() and np.all((scores>=0)&(scores<=1))
    assert all(len(set(map(int,r)))==5 for r in rows)
    assert np.all(kb_source_ids[rows]!=np.asarray(TEST_IDS)[:,None])
    SOURCE_RECORDS[seed]={"work":work,"checkpoint":paths["best_model.pt"],"manifest":manifest,"log":log,"config":cfg,
                          "cache_path":cache_path,"contract_sha256":contract}
    SOURCE_CACHES[seed]={"row_indices":np.array(rows,copy=True),"scores":np.asarray(scores,dtype=np.float32).copy()}
    print(f"Nguồn seed {seed}: train đủ 15 epoch, best epoch {log['best_epoch']}, Val w=.6/lambda=.7, cache Test hợp lệ.")
selected_cache=SOURCE_CACHES[SELECTED_SEED]
test_knowledge=np.array(kb[selected_cache["row_indices"]],dtype=np.float32,copy=True)
test_scores=selected_cache["scores"]
assert test_knowledge.shape==(5000,5,512) and np.isfinite(test_knowledge).all()
assert np.allclose(np.linalg.norm(test_knowledge,axis=-1),1.,atol=1e-3)
# Đọc lại fingerprint sau khi chụp input vào RAM để phát hiện thay đổi giữa chừng.
for path in common_inputs:
    rel=path.relative_to(DATA_ROOT).as_posix()
    if sha256_file(path)!=SOURCE_FILE_HASHES[rel]:
        raise RuntimeError(f"Nguồn thay đổi trong lúc nạp snapshot; chưa tạo output: {rel}")
# Chỉ những embedding đã kiểm tra được giữ trong RAM cho seed đang chạy.
del kb, kb_source_ids
_TOKENIZED_REFERENCES=None
print(f"Sẵn sàng seed {SELECTED_SEED}: 5.000 ảnh, 25.010 references, GPU {torch.cuda.get_device_name()}.")
print("Toàn bộ checkpoint/cache/manifest/kết quả cũ chỉ được đọc. Chưa tạo output mới.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kiểm tra SHA-256: baseline_level1/vocab.json
Kiểm tra SHA-256: COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json
Kiểm tra SHA-256: baseline_level1/clip_image_features/test_features.npy
Kiểm tra SHA-256: baseline_level1/clip_image_features/test_image_ids.json
Kiểm tra SHA-256: baseline_level1/clip_image_features/train_image_ids.json
Kiểm tra SHA-256: knowledge_base_level1_karpathy/text_embeddings_train_restval.npy
Kiểm tra SHA-256: knowledge_base_level1_karpathy/text_embedding_mapping.jsonl
Kiểm tra SHA-256: baseline_level5/lv5_A/trained_5tokens_v1/seed42/run_manifest.json
Kiểm tra SHA-256: baseline_level5/lv5_A/trained_5tokens_v1/seed42/training_log.json
Kiểm tra SHA-256: baseline_level5/lv5_A/trained_5tokens_v1/seed42/final_config.json
Kiểm tra SHA-256: baseline_level5/lv5_A/trained_5tokens_v1/seed42/best_model.pt
Kiểm tra SHA-256: b

In [8]:
#@title CELL 3 — Model native 5 tokens và Beam Search giống Level 1
def create_level5_native(stoi, device):
    VOCAB_SIZE = len(stoi)
    PAD_ID, BOS_ID, EOS_ID = (stoi[x] for x in ('<PAD>', '<BOS>', '<EOS>'))
    D_MODEL, NUM_LAYERS, NUM_HEADS, DIM_FEEDFORWARD = 256, 3, 8, 1024
    DROPOUT, MAX_LENGTH, TAU, BEAM_SIZE, BEAM_ALPHA = 0.1, 30, 1.0, 3, 0.6
    itos = {v: k for k, v in stoi.items()}
    def decode_ids(ids):
        words = []
        for raw in ids:
            word = itos.get(int(raw), '<UNK>')
            if word == '<EOS>':
                break
            if word not in {'<PAD>', '<BOS>'}:
                words.append(word)
        return ' '.join(words).strip()

    class GatedFusion(nn.Module):
        """1. CƠ CHẾ GATED FUSION:
        Sinh hệ số cổng vô hướng g ∈ [0, 1] từ ảnh và trung bình tri thức.
        Đầu vào: [v_norm (512); k_agg_norm (512); confidence (1)] = 1025 chiều.
        Tầng Linear: Linear(1025, 1) với đúng 1,026 tham số.
        Zero-Init (W=0, b=0) đảm bảo g ban đầu = 0.5000 chính xác!"""
        def __init__(self, clip_dim=512):
            super().__init__()
            self.gate_linear = nn.Linear(clip_dim * 2 + 1, 1)
            nn.init.zeros_(self.gate_linear.weight)
            nn.init.zeros_(self.gate_linear.bias)
            self.sigmoid = nn.Sigmoid()

        def forward(self, v_img, k_tokens, scores):
            v_norm = F.normalize(v_img, p=2, dim=-1)
            # Softmax pooling trên 5 câu truy hồi
            w = F.softmax(scores / TAU, dim=-1).unsqueeze(1) # [B, 1, 5]
            k_agg = torch.bmm(w, k_tokens).squeeze(1)        # [B, 512]
            k_agg_norm = F.normalize(k_agg, p=2, dim=-1)
            conf = scores.mean(dim=-1, keepdim=True)         # [B, 1]

            gate_input = torch.cat([v_norm, k_agg_norm, conf], dim=-1) # [B, 1025]
            g = self.sigmoid(self.gate_linear(gate_input))             # [B, 1]
            return g

    class DualBranchDecoderLayer(nn.Module):
        """2. CƠ CHẾ MULTI-TOKEN DUAL-BRANCH DECODER:
        - Nhánh Visual Cross-Attention: Chú ý vào 1 token ảnh (m_vis).
        - Nhánh Knowledge Cross-Attention: nhận 5 tokens từ memory Level 5A.
        - Dung hợp tại tầng ngữ cảnh: c_fused = (1 - g) * c_vis + g * c_knw."""
        def __init__(self, d_model=256, nhead=8, dim_feedforward=1024, dropout=0.1):
            super().__init__()
            self.self_attn  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            self.cross_vis  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            self.cross_knw  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

            self.linear1    = nn.Linear(d_model, dim_feedforward)
            self.linear2    = nn.Linear(dim_feedforward, d_model)
            self.activation = nn.ReLU()

            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)
            self.drop  = nn.Dropout(dropout)

        def forward(self, tgt, m_vis, m_knw, gate, tgt_mask=None, tgt_pad_mask=None):
            # 1. Self-Attention
            h_self = self.norm1(tgt + self.drop(self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_pad_mask)[0]))
            # 2. Dual Cross-Attentions
            c_vis = self.cross_vis(h_self, m_vis, m_vis)[0] # Ngữ cảnh thị giác
            c_knw = self.cross_knw(h_self, m_knw, m_knw)[0] # Ngữ cảnh tri thức đa token
            # 3. Gated Blending
            g_expand = gate.unsqueeze(1) # [B, 1, 1]
            c_fused  = (1.0 - g_expand) * c_vis + g_expand * c_knw
            h_cross  = self.norm2(h_self + self.drop(c_fused))
            # 4. Feed-Forward Network
            out = self.norm3(h_cross + self.drop(self.linear2(self.drop(self.activation(self.linear1(h_cross))))))
            return out

    class Level5_RAGCaptioner(nn.Module):
        """Giữ core Decoder L4; một memory token cho mỗi caption trong 5 caption L5A."""
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_layers=NUM_LAYERS, nhead=NUM_HEADS, d_ff=DIM_FEEDFORWARD, dropout=DROPOUT, max_len=MAX_LENGTH):
            super().__init__()
            self.token_embedding   = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
            self.position_embedding = nn.Embedding(max_len, d_model)

            self.visual_proj       = nn.Linear(512, d_model)
            self.knowledge_proj    = nn.Linear(512, d_model)
            self.knowledge_memory_tokens = 5
            self.tokens_per_caption = 1
            self.d_model = d_model
            self.gated_fusion      = GatedFusion(clip_dim=512)

            self.layers = nn.ModuleList([DualBranchDecoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
            self.output_projection = nn.Linear(d_model, vocab_size)

        def build_knowledge_memory(self, caption_embeddings):
            if caption_embeddings.ndim != 3 or caption_embeddings.shape[1:] != (5, 512):
                raise ValueError('MMR đầu vào phải là 5 caption embeddings [B,5,512]')
            B = caption_embeddings.shape[0]
            projected = self.knowledge_proj(caption_embeddings)  # [B,5,d_model]
            memory = projected
            assert memory.shape == (B, 5, self.d_model)
            return memory

        def forward(self, v_img, k_tokens, scores, input_ids):
            B, T = input_ids.shape
            if k_tokens.shape != (B, 5, 512) or scores.shape != (B, 5):
                raise ValueError('Cache/Gate cần [B,5,512] caption embeddings và [B,5] scores')
            g = self.gated_fusion(v_img, k_tokens, scores) # [B, 1]

            m_vis = self.visual_proj(v_img).unsqueeze(1)   # [B, 1, 256]
            m_knw = self.build_knowledge_memory(k_tokens) # [B,5,256] tới mọi cross_knw layer

            pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)
            h = self.token_embedding(input_ids) + self.position_embedding(pos)
            mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=input_ids.device), diagonal=1)
            pad_mask = input_ids.eq(PAD_ID)

            for layer in self.layers:
                h = layer(h, m_vis, m_knw, g, tgt_mask=mask, tgt_pad_mask=pad_mask)

            logits = self.output_projection(h)
            return logits, g

    return Level5_RAGCaptioner().to(device)

def state_dict_sha256(state):
    h = hashlib.sha256()
    for name in sorted(state):
        value = state[name].detach().cpu().contiguous()
        h.update(json.dumps([name, str(value.dtype), list(value.shape)], separators=(',', ':')).encode())
        h.update(value.view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

@torch.inference_mode()
def beam_search_decode(model, image_feature, knowledge, scores, beam_size=3, max_len=30, alpha=0.6):
    """max_len gồm BOS; mỗi bước xếp hạng cả beam EOS và beam đang mở rộng."""
    if beam_size < 1 or not 2 <= max_len <= model.position_embedding.num_embeddings:
        raise ValueError("beam_size >= 1 và 2 <= max_len <= 30.")
    model.eval()
    device = next(model.parameters()).device
    image_feature = torch.as_tensor(image_feature, dtype=torch.float32, device=device)
    if image_feature.ndim == 2 and image_feature.shape[0] == 1:
        image_feature = image_feature[0]
    if image_feature.shape != (512,):
        raise ValueError(f"Cần một feature 512 chiều, nhận {image_feature.shape}.")
    knowledge = torch.as_tensor(knowledge, dtype=torch.float32, device=device)
    scores = torch.as_tensor(scores, dtype=torch.float32, device=device)
    if knowledge.shape != (5,512) or scores.shape != (5,):
        raise ValueError("Knowledge cần [5,512], scores cần [5].")
    if not all(torch.isfinite(x).all() for x in (image_feature,knowledge,scores)):
        raise ValueError("Input decode có NaN/Inf.")
    beams = [([BOS_ID], 0.0)]
    def normalized(beam):
        tokens, score = beam
        return score / (((5.0 + len(tokens) - 1) / 6.0) ** alpha)
    for _ in range(max_len - 1):
        active = [(tokens, score) for tokens, score in beams if tokens[-1] != EOS_ID]
        if not active:
            break
        candidates = [(tokens, score) for tokens, score in beams if tokens[-1] == EOS_ID]
        inputs = torch.tensor([tokens for tokens, _ in active], device=device)
        memory = image_feature.unsqueeze(0).expand(len(active), -1)
        knw = knowledge.unsqueeze(0).expand(len(active), -1, -1)
        rel = scores.unsqueeze(0).expand(len(active), -1)
        logits = model(memory, knw, rel, inputs)[0][:, -1, :].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError("Model sinh logits không hữu hạn; dừng để kiểm tra checkpoint.")
        logits[:, [PAD_ID, BOS_ID]] = -torch.inf
        log_prob = torch.log_softmax(logits, dim=-1)
        top_scores, top_ids = log_prob.topk(min(beam_size, VOCAB_SIZE - 2), dim=-1)
        # Một lần chuyển GPU -> CPU cho toàn bộ beam ở bước này.
        for (tokens, score), values, indices in zip(
                active, top_scores.tolist(), top_ids.tolist()):
            candidates.extend((tokens + [int(token)], score + float(value))
                              for value, token in zip(values, indices))
        beams = sorted(candidates, key=normalized, reverse=True)[:beam_size]
        if all(tokens[-1] == EOS_ID for tokens, _ in beams):
            break
    best_tokens, _ = max(beams, key=normalized)
    return " ".join(itos[t] for t in best_tokens if t not in (PAD_ID, BOS_ID, EOS_ID))

def load_original_model(seed):
    record=SOURCE_RECORDS[seed]
    path=record["checkpoint"]
    expected_file=SOURCE_FILE_HASHES[path.relative_to(DATA_ROOT).as_posix()]
    if sha256_file(path)!=expected_file:
        raise RuntimeError("Checkpoint nguồn vừa đổi; giữ mọi file và dừng.")
    # TorchVersion là kiểu metadata duy nhất được cho phép thêm; không fallback pickle tùy ý.
    with torch.serialization.safe_globals([torch.torch_version.TorchVersion]):
        checkpoint=torch.load(path,map_location="cpu",weights_only=True)
    if sha256_file(path)!=expected_file:
        raise RuntimeError("Checkpoint đổi trong lúc nạp.")
    contract,evidence=checkpoint["contract"],checkpoint["evidence"]
    assert contract["seed"]==evidence["training_seed"]==seed
    assert contract["contract_sha256"]==record["contract_sha256"]
    assert contract["architecture"]=="level5_native5_trained_v1" and contract["memory_contract"]==MEMORY_CONTRACT
    assert evidence["trained"] is True and evidence["completed_epochs"]==15 and evidence["optimizer_steps"]==132840
    assert evidence["best_epoch"]==checkpoint["epoch"]==record["log"]["best_epoch"]
    assert math.isclose(float(checkpoint["val_cider"]),float(record["log"]["best_val_cider"]),rel_tol=0,abs_tol=1e-12)
    assert evidence["initial_state_sha256"]==EXPECTED_INIT_STATE
    assert contract["init_checkpoint_sha256"]==EXPECTED_INIT_FILE
    assert contract["code_sha256"]==record["manifest"]["code_sha256"]
    state=checkpoint["model_state_dict"]
    assert state and all(isinstance(v,torch.Tensor) and torch.isfinite(v).all() for v in state.values())
    actual_state=state_dict_sha256(state)
    assert actual_state==evidence["model_state_sha256"]==record["config"]["checkpoint_sha256"]==EXPECTED_STATES[seed]
    model=create_level5_native(stoi,DEVICE)
    model.load_state_dict(state,strict=True)
    assert model.knowledge_proj.weight.shape==(256,512) and model.knowledge_memory_tokens==5
    assert state_dict_sha256(model.state_dict())==actual_state
    del checkpoint,state
    return model.eval()

DECODE_CONFIG=dict(method="beam_search",beam_size=3,alpha=0.6,max_len=MAX_LEN,
    max_len_includes_bos=True,mask_pad_bos=True,implementation="normalized_candidates_v1",precision="float32",tf32=False)
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
PROTOCOL=dict(level="L5A_native5",purpose="re-evaluation of existing trained checkpoints; no training",
    seeds=ALL_SEEDS,experiment_tag=EXPERIMENT_TAG,decode=DECODE_CONFIG,
    evaluator="pycocoevalcap==1.2/PTB/Java11",metric_scale="raw pycocoevalcap",
    memory_contract=MEMORY_CONTRACT,expected_model_states={str(k):v for k,v in EXPECTED_STATES.items()},
    original_files_sha256=SOURCE_FILE_HASHES,selection="frozen existing validation w=.6/lambda=.7; no test tuning",
    model_implementation="embedded native5 architecture v1; strict keys/shapes/dtypes and original state SHA",
    storage_schema="guarded_l5a_reevaluation_v1")
PROTOCOL["notebook_logic_sha256"]="c939352be8c4e17a1279c489ac47faf74ef4fc013a004eecc5e92a460b978edf"
PROTOCOL_HASH=canonical_hash(PROTOCOL)

def assert_sources_unchanged(seed=None):
    # KB/feature snapshot đã ở RAM. Xác minh metadata/checkpoint/cache nguồn trước khi lưu kết quả.
    for rel,expected in SOURCE_FILE_HASHES.items():
        if rel.endswith(".npy") or rel.endswith("text_embedding_mapping.jsonl"):
            continue
        if seed is not None and "/seed" in rel and f"/seed{seed}/" not in rel:
            continue
        if sha256_file(SOURCE_FILE_PATHS[rel])!=expected:
            raise RuntimeError(f"Nguồn vừa đổi; không tiếp tục/ghi kết quả lẫn dữ liệu: {rel}")

torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
torch.use_deterministic_algorithms(True)
_model_check=load_original_model(SELECTED_SEED)
with torch.inference_mode():
    _logits,_gate=_model_check(torch.as_tensor(test_features[:1],device=DEVICE),
        torch.as_tensor(test_knowledge[:1],device=DEVICE),torch.as_tensor(test_scores[:1],device=DEVICE),
        torch.tensor([[BOS_ID]],device=DEVICE))
    assert _logits.shape==(1,1,VOCAB_SIZE) and torch.isfinite(_logits).all() and _gate.shape==(1,1)
del _model_check,_logits,_gate
gc.collect();torch.cuda.empty_cache()
print("Strict-load model seed",SELECTED_SEED,"OK • protocol",PROTOCOL_HASH[:16])


Strict-load model seed 44 OK • protocol 1038ce1f037c038b


In [9]:
#@title CELL 4 — Khởi tạo vùng kết quả mới, bảo vệ file và khóa từng seed
# Mọi đường dẫn ghi đều bị giới hạn vào RUN_DIR; input gốc chỉ được đọc.
import uuid
from contextlib import contextmanager
RUN_DIR = OUTPUT_ROOT / ("run_" + PROTOCOL_HASH[:16])
_OBSERVED_FILES = {}
_ACTIVE_LOCKS = {}

def output_path(path):
    path = Path(path).resolve()
    try:
        path.relative_to(RUN_DIR.resolve())
    except ValueError as exc:
        raise PermissionError(f"Từ chối ghi ngoài vùng kết quả mới: {path}") from exc
    return path

def owned_meta(seed, role):
    return {"protocol_hash": PROTOCOL_HASH, "storage_schema": 2,
            "seed": seed, "role": role}

def verify_owner(payload, seed, role):
    expected = owned_meta(seed, role)
    meta = payload.get("_meta", {}) if isinstance(payload, dict) else {}
    if any(meta.get(k) != v for k, v in expected.items()):
        raise ValueError(f"File khác seed/protocol/định dạng ({role}); giữ nguyên và dừng.")

def remember_file(path, before):
    path = output_path(path)
    after = sha256_file(path)
    if before != after:
        raise RuntimeError(f"File đang bị phiên khác thay đổi: {path.name}")
    _OBSERVED_FILES[str(path)] = after

def read_owned_json(path, seed, role):
    path = output_path(path)
    if not path.exists():
        _OBSERVED_FILES[str(path)] = None
        return None
    if not path.is_file():
        raise FileExistsError(f"Đường dẫn đã là thư mục: {path}")
    before = sha256_file(path)
    payload = read_json(path)  # JSON lỗi: dừng, không tự ghi đè.
    verify_owner(payload, seed, role)
    remember_file(path, before)
    return payload

def seed_dir(seed, create=False):
    if seed not in (42, 44, 46):
        raise ValueError("Chỉ có seed 42, 44, 46.")
    path = output_path(RUN_DIR / f"seed_{seed}")
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path

def check_lock(scope):
    lock = _ACTIVE_LOCKS.get(scope)
    if lock is None:
        raise RuntimeError(f"Thiếu khóa ghi cho {scope}.")
    path, token = lock
    try:
        current = read_json(path)
    except (OSError, ValueError) as exc:
        raise RuntimeError("Khóa bị mất hoặc hỏng; dừng ghi.") from exc
    if current.get("token") != token:
        raise RuntimeError("Quyền ghi đã thay đổi; dừng để bảo vệ dữ liệu.")

@contextmanager
def writing_scope(scope):
    global OLD_SESSION_STOPPED
    # Một phiên đang chạy cho mỗi seed. Khóa trên Drive là lớp phòng vệ,
    # không thay thế quy tắc không chạy hai Colab VM cùng seed.
    folder = RUN_DIR if scope == "summary" else seed_dir(scope, create=True)
    lock_path = output_path(folder / ".active_run.json")
    if scope in _ACTIVE_LOCKS:
        raise RuntimeError(f"Seed/phần tổng hợp {scope} đang chạy trong phiên này.")
    if lock_path.exists():
        old_hash = sha256_file(lock_path)
        try:
            old = read_json(lock_path)
        except (OSError, ValueError):
            old = None
        if not OLD_SESSION_STOPPED:
            raise RuntimeError(
                f"Còn phiên/khóa cũ của {scope}. Nếu phiên cũ đã DỪNG HẲN, bật "
                "OLD_SESSION_STOPPED ở Cell 2, rồi chạy lại. Không chạy hai phiên cùng seed.")
        if old is not None:
            verify_owner(old, scope, "active_run")
        elif lock_path.stat().st_size > 4096:
            raise RuntimeError("File khóa hỏng có kích thước bất thường; giữ nguyên để kiểm tra.")
        if sha256_file(lock_path) != old_hash:
            raise RuntimeError("Khóa vừa đổi; không thu hồi.")
        lock_path.unlink()
        OLD_SESSION_STOPPED = False  # Thu hồi một lần, tránh vô tình lấy khóa khác.
    token = uuid.uuid4().hex
    record = {"_meta": owned_meta(scope, "active_run"), "token": token,
              "started_unix": time.time()}
    with lock_path.open("x", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False)
    _ACTIVE_LOCKS[scope] = (lock_path, token)
    try:
        check_lock(scope)
        yield
    finally:
        try:
            if lock_path.is_file() and read_json(lock_path).get("token") == token:
                lock_path.unlink()
        finally:
            _ACTIVE_LOCKS.pop(scope, None)

def required_scope(path):
    relative = output_path(path).relative_to(RUN_DIR.resolve())
    if len(relative.parts) > 1 and relative.parts[0].startswith("seed_"):
        return int(relative.parts[0].split("_")[1])
    return "summary"

def commit_file(path, write_temp, mutable=False):
    path = output_path(path)
    scope = required_scope(path)
    check_lock(scope)
    key = str(path)
    if key not in _OBSERVED_FILES:
        if path.exists():
            raise FileExistsError(f"Chưa xác minh file có sẵn; KHÔNG ghi đè: {path}")
        _OBSERVED_FILES[key] = None
    expected = _OBSERVED_FILES[key]
    current = sha256_file(path) if path.exists() else None
    if current != expected:
        raise RuntimeError(f"File đã đổi sau khi đọc; KHÔNG ghi đè: {path}")
    tmp = output_path(path.with_name("." + path.name + "." + uuid.uuid4().hex + ".tmp"))
    try:
        write_temp(tmp)
        new_hash = sha256_file(tmp)
        if current == new_hash:
            return False
        if current is not None and not mutable:
            raise FileExistsError(f"Kết quả đã có và khác nội dung; giữ nguyên: {path}")
        check_lock(scope)
        if (sha256_file(path) if path.exists() else None) != expected:
            raise RuntimeError(f"File vừa đổi; KHÔNG ghi đè: {path}")
        # Chỉ thay trạng thái đang chạy của đúng experiment/seed đã xác minh.
        # Không di chuyển hay đổi tên bất kỳ dữ liệu/checkpoint nguồn nào.
        os.replace(tmp, path)
        _OBSERVED_FILES[key] = new_hash
        if sha256_file(path) != new_hash:
            raise IOError(f"Readback sau lưu không khớp: {path}")
        return True
    finally:
        # Chỉ xóa file tạm UUID của chính lần ghi này.
        tmp.unlink(missing_ok=True)

def save_json(value, path, mutable=False):
    def write_temp(tmp):
        with tmp.open("x", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False, sort_keys=True,
                      indent=2, allow_nan=False)
            f.flush()
            os.fsync(f.fileno())
    return commit_file(path, write_temp, mutable=mutable)

def initialize_experiment():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = RUN_DIR / "experiment.json"
    manifest = {"_meta": owned_meta(None, "experiment"),
                "protocol": PROTOCOL, "seeds": [42, 44, 46]}
    def validate_manifest():
        last_error = None
        for attempt in range(6):
            try:
                previous = read_owned_json(manifest_path, None, "experiment")
                if previous is not None:
                    if previous != manifest:
                        raise ValueError("Manifest khác cấu hình; không dùng chung thư mục.")
                    return
            except (OSError, json.JSONDecodeError) as exc:
                last_error = exc
            if attempt < 5:
                time.sleep(0.2)
        raise RuntimeError(
            "Manifest thiếu/hỏng hoặc chưa ghi xong; mọi file được giữ nguyên. "
            "Thử chạy lại Cell 3. Nếu vẫn lỗi, đổi EXPERIMENT_TAG ở Cell 2 để "
            "tạo thí nghiệm mới, không xóa/ghi đè thư mục cũ.") from last_error

    if manifest_path.exists():
        validate_manifest()
    elif any(RUN_DIR.iterdir()):
        # Seed khác có thể vừa khởi tạo manifest giữa exists() và iterdir().
        validate_manifest()
    else:
        try:
            with manifest_path.open("x", encoding="utf-8") as f:
                json.dump(manifest, f, ensure_ascii=False, sort_keys=True, indent=2)
                f.flush()
                os.fsync(f.fileno())
        except FileExistsError:
            validate_manifest()
    print("Kết quả mới:", RUN_DIR)
    print(f"Seed đang chọn: {SELECTED_SEED} →", seed_dir(SELECTED_SEED))
    print("Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.")


assert_sources_unchanged(SELECTED_SEED)
initialize_experiment()


Kết quả mới: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level5/lv5_A/multiseed_beam3_v1/run_1038ce1f037c038b
Seed đang chọn: 44 → /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level5/lv5_A/multiseed_beam3_v1/run_1038ce1f037c038b/seed_44
Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.


In [10]:
#@title CELL 5 — Đánh giá seed đang chọn; tự tiếp tục predictions và từng metric
def prediction_prefix_valid(predictions, complete=False):
    if not isinstance(predictions, list) or len(predictions) > len(image_ids["test"]):
        return False
    try:
        ids = [int(row["image_id"]) for row in predictions]
        valid = (ids == image_ids["test"][:len(ids)]
                 and len(ids) == len(set(ids))
                 and all(isinstance(row["caption"], str) for row in predictions))
        return valid and (not complete or len(ids) == 5000)
    except (KeyError, TypeError, ValueError):
        return False

def valid_metrics(metrics):
    return all(name in metrics and isinstance(metrics[name], (int, float))
               and math.isfinite(float(metrics[name])) for name in METRIC_NAMES)

def load_prediction_packet(path, seed, signature):
    packet = read_owned_json(path, seed, "predictions")
    if packet is None:
        return None
    meta = packet["_meta"]
    predictions = packet.get("predictions")
    if meta.get("signature") != signature:
        raise ValueError("Predictions khác checkpoint/cấu hình; GIỮ NGUYÊN, không ghi đè.")
    complete = meta.get("complete")
    if not isinstance(complete, bool) or not prediction_prefix_valid(predictions, complete):
        raise ValueError("Predictions hỏng/trùng/sai thứ tự ID; giữ nguyên và dừng.")
    if meta.get("predictions_hash") != canonical_hash(predictions):
        raise ValueError("Nội dung predictions đã đổi; giữ nguyên và dừng.")
    return packet

def score_predictions(predictions, seed, signature):
    global _TOKENIZED_REFERENCES
    if not prediction_prefix_valid(predictions,complete=True):
        raise ValueError("Predictions phải phủ đúng 5.000 test IDs.")
    progress_path=seed_dir(seed)/f"metric_progress_seed{seed}.json"
    progress=read_owned_json(progress_path,seed,"metric_progress")
    prediction_hash=canonical_hash(predictions)
    metrics={}
    if progress is not None:
        if progress["_meta"].get("signature")!=signature or progress.get("predictions_hash")!=prediction_hash:
            raise ValueError("Tiến độ metric khác predictions; giữ nguyên và dừng.")
        metrics=dict(progress.get("metrics",{}))
        if not set(metrics)<=set(METRIC_NAMES) or not all(isinstance(x,(float,int)) and math.isfinite(float(x)) for x in metrics.values()):
            raise ValueError("Tiến độ metric hỏng; không tự ghi đè.")
    if valid_metrics(metrics):
        print("Đã có đủ 8 metric hợp lệ; không chấm lại.")
        return metrics
    if _TOKENIZED_REFERENCES is None:
        print("PTB tokenize 25.010 reference captions...")
        _TOKENIZED_REFERENCES=PTBTokenizer().tokenize({i:[{"caption":c} for c in references[i]] for i in TEST_IDS})
    pred_by_id={int(r["image_id"]):r["caption"] for r in predictions}
    res=PTBTokenizer().tokenize({i:[{"caption":pred_by_id[i]}] for i in TEST_IDS})
    scorers=[(lambda:Bleu(4),METRIC_NAMES[:4]),(ManagedMeteor,["METEOR"]),(Rouge,["ROUGE_L"]),(Cider,["CIDEr"]),(Spice,["SPICE"])]
    for factory,names in scorers:
        if all(name in metrics for name in names):
            print("Dùng metric đã lưu:",", ".join(names));continue
        print("Đang chấm:",", ".join(names),flush=True)
        scorer=factory()
        try:
            score,_=scorer.compute_score(_TOKENIZED_REFERENCES,res)
            for name,value in zip(names,score if len(names)>1 else [score]):
                value=float(value)
                if not math.isfinite(value):raise ValueError(f"{name} không hữu hạn.")
                metrics[name]=value
        except Exception as exc:
            raise RuntimeError(f"Chấm {names} thất bại; giữ predictions và metric đã lưu: {exc}") from exc
        finally:
            if isinstance(scorer,ManagedMeteor):scorer.close()
            del scorer;gc.collect()
        updated={"_meta":{**owned_meta(seed,"metric_progress"),"signature":signature},
                 "predictions_hash":prediction_hash,"metrics":dict(metrics)}
        save_json(updated,progress_path,mutable=progress is not None)
        progress=updated
    if not valid_metrics(metrics):raise RuntimeError("Chưa đủ 8 metric; không điền số thay thế.")
    return metrics

def evaluate_seed(seed):
    check_lock(seed)
    assert seed==SELECTED_SEED, "Đổi seed ở Cell 2 rồi chạy lại Cell 2–6 để nạp đúng cache."
    assert_sources_unchanged(seed)
    checkpoint_path=SOURCE_RECORDS[seed]["checkpoint"]
    checkpoint_hash=SOURCE_FILE_HASHES[checkpoint_path.relative_to(DATA_ROOT).as_posix()]
    signature=canonical_hash({"seed":seed,"checkpoint_sha256":checkpoint_hash,"protocol_hash":PROTOCOL_HASH})
    predictions_path=seed_dir(seed)/f"predictions_seed{seed}.json"
    metrics_path=seed_dir(seed)/f"metrics_seed{seed}.json"
    packet=load_prediction_packet(predictions_path,seed,signature)
    previous=read_owned_json(metrics_path,seed,"metrics")
    if previous is not None:
        if (previous["_meta"].get("signature")!=signature or previous["_meta"].get("status")!="complete"
                or not valid_metrics(previous) or packet is None or not packet["_meta"]["complete"]
                or previous["_meta"].get("predictions_sha256")!=sha256_file(predictions_path)):
            raise ValueError("Kết quả có sẵn không xác minh được; GIỮ NGUYÊN và dừng.")
        print(f"Seed {seed}: đủ kết quả cuối hợp lệ; không sửa file.")
        return previous
    predictions=[] if packet is None else list(packet["predictions"])
    started=time.perf_counter()
    def persist(complete):
        nonlocal packet
        if packet is not None and packet["_meta"]["complete"]:
            raise FileExistsError("Predictions đã hoàn tất; giữ nguyên.")
        if not prediction_prefix_valid(predictions,complete):
            raise ValueError("Predictions trùng/sai thứ tự/thiếu IDs.")
        updated={"_meta":{**owned_meta(seed,"predictions"),"signature":signature,"complete":complete,
            "checkpoint_sha256":checkpoint_hash,"num_images":len(predictions),
            "predictions_hash":canonical_hash(predictions),"format":"envelope; COCO rows in predictions key"},
            "predictions":list(predictions)}
        save_json(updated,predictions_path,mutable=packet is not None)
        packet={"_meta":dict(updated["_meta"]),"predictions":list(predictions)}
    if packet is None or not packet["_meta"]["complete"]:
        model=load_original_model(seed)
        last_saved=len(predictions)
        print(f"Seed {seed}: tiếp tục từ ảnh {len(predictions)+1}/5000.")
        try:
            with tqdm(total=5000,initial=len(predictions),desc=f"L5A seed {seed} • Beam3",unit="ảnh") as bar:
                for position in range(len(predictions),5000):
                    caption=beam_search_decode(model,test_features[position],test_knowledge[position],test_scores[position],
                        beam_size=3,max_len=MAX_LEN,alpha=0.6)
                    predictions.append({"image_id":TEST_IDS[position],"caption":caption})
                    bar.update(1)
                    if len(predictions)%100==0 and len(predictions)<5000:
                        persist(False);last_saved=len(predictions)
            assert_sources_unchanged(seed)
            persist(True)
        except BaseException:
            if last_saved<len(predictions)<5000:
                try:persist(False)
                except Exception as save_error:print("Không lưu thêm được; bản lưu trước được giữ:",save_error)
            raise
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    else:
        print("Đã đủ 5.000 predictions; chỉ tiếp tục chấm điểm.")
    prediction_hash=sha256_file(predictions_path)
    metrics=score_predictions(predictions,seed,signature)
    assert_sources_unchanged(seed)
    if sha256_file(predictions_path)!=prediction_hash:
        raise RuntimeError("Predictions vừa đổi trong lúc chấm; không ghi metrics cuối.")
    metrics["_meta"]={**owned_meta(seed,"metrics"),"signature":signature,"status":"complete",
        "checkpoint":str(checkpoint_path),"checkpoint_sha256":checkpoint_hash,"model_state_sha256":EXPECTED_STATES[seed],
        "predictions_sha256":prediction_hash,"decoding":DECODE_CONFIG,"split":"COCO2014 Karpathy Test",
        "num_images":5000,"num_references":25010,"metric_scale":"raw pycocoevalcap scores; x100 for display",
        "frozen_validation_config":{"w":0.6,"lambda_param":0.7},"trained":True,"training_seed":seed,
        "elapsed_seconds_this_session":time.perf_counter()-started,
        "environment":{"python":platform.python_version(),"torch":str(torch.__version__),"numpy":np.__version__,
            "pycocoevalcap":metadata.version("pycocoevalcap"),"gpu":torch.cuda.get_device_name(),"java":"11"}}
    save_json(metrics,metrics_path)
    print(f"Seed {seed} hoàn tất (×100):",{k:round(metrics[k]*100,4) for k in METRIC_NAMES})
    return metrics

with writing_scope(SELECTED_SEED):
    selected_metrics=evaluate_seed(SELECTED_SEED)


Seed 44: tiếp tục từ ảnh 1/5000.


L5A seed 44 • Beam3:   0%|          | 0/5000 [00:00<?, ?ảnh/s]

PTB tokenize 25.010 reference captions...
Đang chấm: Bleu_1, Bleu_2, Bleu_3, Bleu_4
{'testlen': 48099, 'reflen': 47701, 'guess': [48099, 43099, 38099, 33099], 'correct': [35753, 19531, 10021, 5122]}
ratio: 1.0083436405945156
Đang chấm: METEOR
Đang chấm: ROUGE_L
Đang chấm: CIDEr
Đang chấm: SPICE
Seed 44 hoàn tất (×100): {'Bleu_1': 74.3321, 'Bleu_2': 58.0386, 'Bleu_3': 44.5804, 'Bleu_4': 34.2187, 'METEOR': 27.5237, 'ROUGE_L': 55.7995, 'CIDEr': 111.4619, 'SPICE': 20.2954}


In [11]:
#@title CELL 6 — Tổng hợp đúng ba seed khi đủ kết quả mới
def summarize_completed_seeds():
    rows,source_hashes={},{}
    assert_sources_unchanged()
    for seed in ALL_SEEDS:
        metric_path=seed_dir(seed)/f"metrics_seed{seed}.json"
        prediction_path=seed_dir(seed)/f"predictions_seed{seed}.json"
        result=read_owned_json(metric_path,seed,"metrics")
        if result is None or not valid_metrics(result):raise ValueError(f"Seed {seed} chưa đủ 8 metrics.")
        meta=result["_meta"]
        checkpoint=SOURCE_RECORDS[seed]["checkpoint"]
        checkpoint_hash=SOURCE_FILE_HASHES[checkpoint.relative_to(DATA_ROOT).as_posix()]
        signature=canonical_hash({"seed":seed,"checkpoint_sha256":checkpoint_hash,"protocol_hash":PROTOCOL_HASH})
        assert meta["signature"]==signature and meta["status"]=="complete" and meta["seed"]==seed
        assert meta["model_state_sha256"]==EXPECTED_STATES[seed] and meta["checkpoint_sha256"]==checkpoint_hash
        assert Path(meta["checkpoint"]).resolve()==checkpoint.resolve()
        assert meta["decoding"]==DECODE_CONFIG and meta["frozen_validation_config"]=={"w":0.6,"lambda_param":0.7}
        packet=load_prediction_packet(prediction_path,seed,signature)
        assert packet is not None and packet["_meta"]["complete"]
        assert meta["predictions_sha256"]==sha256_file(prediction_path)
        source_hashes[metric_path]=sha256_file(metric_path)
        source_hashes[prediction_path]=meta["predictions_sha256"]
        rows[seed]=result
    assert len(set(EXPECTED_STATES.values()))==3
    statistics={}
    for name in METRIC_NAMES:
        values=np.asarray([rows[s][name] for s in ALL_SEEDS],dtype=np.float64)
        statistics[name]={"mean":float(values.mean()),"std":float(values.std(ddof=1)),
                          "values":{str(s):float(v) for s,v in zip(ALL_SEEDS,values)}}
    summary={"_meta":owned_meta(None,"summary"),"seeds":ALL_SEEDS,"n_seeds":3,"std_ddof":1,
        "std_definition":"sample standard deviation across seeds; not confidence interval",
        "metric_scale":"raw pycocoevalcap; multiply all scores by100 for display",
        "test_images":5000,"decode":DECODE_CONFIG,"protocol_hash":PROTOCOL_HASH,"protocol":PROTOCOL,
        "metrics_sha256":{str(s):source_hashes[seed_dir(s)/f"metrics_seed{s}.json"] for s in ALL_SEEDS},
        "training_note":"No retraining: existing L5A models,15 epochs each,all initialized from L4 Lan01_seed45",
        "selection_note":"Existing validation-selected w=.6/lambda=.7 frozen; no parameter or seed selection on Test",
        "comparison_note":"Decoder standardized to Lv1 normalized_candidates_v1; old L5A beam outputs remain separate",
        "metrics":statistics}
    for path,expected in source_hashes.items():
        if sha256_file(path)!=expected:raise RuntimeError("Nguồn tổng hợp vừa đổi; không ghi summary.")
    summary_path=RUN_DIR/"level5a_3seeds_beam3_summary.json"
    previous=read_owned_json(summary_path,None,"summary")
    if previous is not None:
        if previous!=summary:raise ValueError("Summary có sẵn khác nội dung; giữ nguyên và dừng.")
        print("Summary đã có và khớp; không sửa.")
    else:save_json(summary,summary_path)
    labels={"Bleu_1":"BLEU-1","Bleu_2":"BLEU-2","Bleu_3":"BLEU-3","Bleu_4":"BLEU-4",
        "METEOR":"METEOR","ROUGE_L":"ROUGE-L","CIDEr":"CIDEr","SPICE":"SPICE"}
    table=["| Chỉ số ×100 | Seed 42 | Seed 44 | Seed 46 | Mean ± Std |","|:--|--:|--:|--:|--:|"]
    for name in METRIC_NAMES:
        st=statistics[name]
        values=[labels[name]]+[f"{rows[s][name]*100:.4f}" for s in ALL_SEEDS]
        values.append(f"{st['mean']*100:.4f} ± {st['std']*100:.4f}")
        table.append("| "+" | ".join(values)+" |")
    display(Markdown("\n".join(table)))
    display(Markdown("JSON lưu thang gốc; bảng ×100. Std mẫu với n=3, ddof=1. Giữ cùng ba seed và protocol khi so sánh các Level. Cả ba L5A đã fine-tune từ cùng L4 seed45; chênh lệch không tách riêng tác dụng Semantic/MMR khỏi huấn luyện thêm."))
    latex=[r"\begin{table}[htbp]",r"\centering",
        r"\caption{Level5A on COCO2014 Karpathy Test,Beam3 $\alpha=0.6$,scores $\times100$,mean $\pm$ sample std ($n=3$).}",
        r"\begin{tabular}{lrrrr}",r"\hline",r"Metric & Seed42 & Seed44 & Seed46 & Mean $\pm$ Std \\",r"\hline"]
    for name in METRIC_NAMES:
        st=statistics[name]
        values=[labels[name]]+[f"{rows[s][name]*100:.4f}" for s in ALL_SEEDS]
        values.append("$"+f"{st['mean']*100:.4f} \\pm {st['std']*100:.4f}"+"$")
        latex.append(" & ".join(values)+r" \\")
    latex += [r"\hline",r"\end{tabular}",r"\end{table}"]
    print("\nLaTeX để sao chép vào báo cáo:\n"+"\n".join(latex))
    print("Đã lưu:",summary_path)

missing=[]
for seed in ALL_SEEDS:
    if not (seed_dir(seed)/f"metrics_seed{seed}.json").is_file() or not (seed_dir(seed)/f"predictions_seed{seed}.json").is_file():
        missing.append(seed)
if missing:
    print("Chưa đủ kết quả mới; đây là trạng thái bình thường. Còn thiếu seed:",missing)
    print("Đổi SEED_TO_RUN ở Cell 2 và chạy lại Cell 2–6, giữ cùng tag. Không xóa thư mục.")
else:
    with writing_scope("summary"):
        summarize_completed_seeds()


Chưa đủ kết quả mới; đây là trạng thái bình thường. Còn thiếu seed: [46]
Đổi SEED_TO_RUN ở Cell 2 và chạy lại Cell 2–6, giữ cùng tag. Không xóa thư mục.
